# Multi-unit handling — how `pump` and `pumpflow` keep units honest

This notebook is a **human-readable walkthrough** of the multi-unit design philosophy
of the workbench, with charts that *demonstrate* the invariants rather than just assert
them.

The division of responsibility (the philosophy):

1. **The library owns the canonical unit.** `pump`'s `STANDARD_UNITS` table is the only
   authority on the magnitude a stored quantity carries (capacity m³/h, head m, power kW…).
2. **The GUI owns the menu of display units.** `pumpflow.units.UNIT_OPTIONS` maps each
   dimension to the display units it offers (label → Pint string).
3. **Display unit ≠ stored unit.** A dialog lets you pick a display unit, but a node's
   `to_signal()` always normalises back to the standard unit via `quantity_factory`.
4. **One conversion path.** Everything routes through `quantity_factory`; viscosity
   (kinematic cSt ↔ dynamic cP, which needs density) is the single documented exception.
5. **Preset, then override.** A field defaults to the active project preset (SI / US /
   custom); an explicit per-field choice wins.

Everything below runs **without Qt** — `pumpflow.units` depends only on `pump`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pump.utilities.unit_conversion import Q_, quantity_factory, STANDARD_UNITS
from pumpflow import units

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True

## 1. The library spine — `quantity_factory` against `STANDARD_UNITS`

Every quantity entering the library is normalised to a fixed standard unit chosen by its
*dimensionality*. The `context` argument disambiguates dimensions that have more than one
engineering convention (e.g. pressure: gauge / absolute / differential).

In [ ]:
print('STANDARD_UNITS (library canonical units):')
for dim, table in STANDARD_UNITS.items():
    print(f'  {dim:18s} -> {table}')

print('\nRound-trips through quantity_factory:')
for q, ctx in [(Q_(500, 'gram'), 'default'),
               (Q_(833, 'm**3/h'), 'default'),
               (Q_(1, 'atm'), 'atm'),
               (Q_(1, 'atm'), 'delta')]:
    print(f'  {str(q):16s} (context={ctx:7s}) -> {quantity_factory(q, ctx)}')

## 2. Conversion-consistency — the same physical duty, many input units

Take one physical capacity, **833 m³/h**, and express it in every display unit the GUI
offers for capacity. The *displayed* magnitudes look wildly different — but once each is
pushed through `units.to_standard(...)` (the exact call a node's `to_signal` makes), they
all collapse back to the same standard magnitude. That collapse is the whole guarantee.

In [ ]:
physical_q_m3h = 833.0
labels = [lab for lab, _ in units.UNIT_OPTIONS['capacity']]

# Express the one physical duty in each display unit...
displayed = [units.convert_display(physical_q_m3h, 'capacity', 'm³/h', lab) for lab in labels]
# ...then normalise each back to the standard unit (what to_signal does).
normalised = [units.to_standard(v, 'capacity', lab) for v, lab in zip(displayed, labels)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(labels, displayed, color='#4C8BF5')
ax1.set_title('Displayed value (per chosen unit)')
ax1.set_ylabel('magnitude as shown')
for i, v in enumerate(displayed):
    ax1.text(i, v, f'{v:.1f}', ha='center', va='bottom', fontsize=9)

ax2.bar(labels, normalised, color='#2BB673')
ax2.axhline(physical_q_m3h, ls='--', color='grey')
ax2.set_title('After to_standard()  →  all identical (m³/h)')
ax2.set_ylabel('standard magnitude [m³/h]')
for i, v in enumerate(normalised):
    ax2.text(i, v, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
fig.suptitle('One physical duty (833 m³/h) — display differs, stored value does not')
plt.tight_layout()
plt.show()

assert max(abs(v - physical_q_m3h) for v in normalised) < 1e-6, 'normalisation must be exact'
print('All display units normalise to', round(normalised[0], 6), 'm³/h — invariant holds.')

## 3. Preset philosophy — one rated point, two unit systems

A project-level preset (`units.PREFS`) chooses the display units a *fresh* field opens in.
Here is one rated duty rendered under the **SI** and **US-customary** presets. The numbers
on screen change; the underlying signal (what every downstream node consumes) does not.

In [ ]:
# One physical rated duty, in standard units.
duty_std = {'capacity': 833.0, 'head': 73.0, 'power': 252.0}
canon = {d: units.canonical_unit(d) for d in duty_std}

def render(preset):
    out = {}
    for dim, std_val in duty_std.items():
        unit = units.PRESETS[preset].get(dim, canon[dim])
        out[dim] = (units.convert_display(std_val, dim, canon[dim], unit), unit)
    return out

si, us = render('SI'), render('US')
print(f"{'quantity':10s} | {'SI display':18s} | {'US display':18s}")
print('-' * 52)
for dim in duty_std:
    print(f"{dim:10s} | {si[dim][0]:8.1f} {si[dim][1]:8s} | {us[dim][0]:8.1f} {us[dim][1]:8s}")

fig, ax = plt.subplots()
x = np.arange(len(duty_std)); w = 0.35
ax.bar(x - w/2, [si[d][0] for d in duty_std], w, label='SI', color='#4C8BF5')
ax.bar(x + w/2, [us[d][0] for d in duty_std], w, label='US customary', color='#F5A623')
ax.set_xticks(x)
ax.set_xticklabels([f"{d}\n({si[d][1]} vs {us[d][1]})" for d in duty_std])
ax.set_yscale('log')
ax.set_ylabel('displayed magnitude (log scale)')
ax.set_title('Same duty, different preset — display only')
ax.legend()
plt.tight_layout()
plt.show()

## 4. The data flow, end to end

```
  dialog field (display unit)            pumpflow.units / ui.UnitField
         │  user types 3667.6 US GPM            registry + conversion
         ▼                                              │
  node.to_signal()  ──  units.to_standard(...)  ────────┘   → 833.0 m³/h
         │  RatedPoint(q_m3h=833.0, …)   ← always standard units
         ▼
  pumpflow.binding  ──  Q_(833.0, 'm**3/h')  →  pump.DesignPoint / PerformanceCurve
```

Where each piece lives:

| Concern | Code |
|---|---|
| Canonical units | `pump/utilities/unit_conversion.py` — `STANDARD_UNITS`, `quantity_factory` |
| Display-unit registry + presets | `pumpflow/units.py` — `UNIT_OPTIONS`, `PRESETS`, `PREFS` |
| Reusable unit-aware field | `pumpflow/nodes/ui.py` — `UnitField` |
| Per-node usage | `pumpflow/nodes/rated_point.py`, `pumpflow/nodes/point.py` |
| Project preset UI + persistence | `pumpflow/app.py` (Units menu), `pumpflow/canvas/scene.py` (`meta`) |

The key invariant, proved by the chart in §2: **no matter which unit the engineer types in,
the signal that flows downstream — and into the `pump` physics — is identical.**